Homework 6: Batch for Data Engineering Zoomcamp 2026

In [7]:
import pyspark
from pyspark.sql import SparkSession
import os
from pyspark.sql import functions as F

In [8]:
import pyspark
from pyspark.sql import SparkSession

# Using local[*] to leverage all 12 cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()

Question 1. Install Spark and PySpark (1 point)
**3.5**

In [9]:
spark.version

'3.5.0'

In [ ]:
# Downloading the November 2025 Yellow Taxi data
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -O yellow_tripdata_2025-11.parquet

--2026-03-05 23:16:03--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.133, 18.239.238.212, 18.239.238.119, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  8.74MB/s    in 7.9s    

2026-03-05 23:16:11 (8.58 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [13]:
# Read the data
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [14]:
# Repartition the Dataframe to 4 partitions and save it to parquet
# Note: This will create the directory data/pq/yellow/2025/11/
df.repartition(4).write.mode("overwrite").parquet('data/pq/yellow/2025/11/')

In [15]:
# Logic to calculate the average size of the .parquet files
path = 'data/pq/yellow/2025/11/'
files = [f for f in os.listdir(path) if f.endswith('.parquet')]
total_size = sum(os.path.getsize(os.path.join(path, f)) for f in files)
avg_size_mb = (total_size / len(files)) / (1024 * 1024)

In [16]:
print(f"Average size of parquet files: {avg_size_mb:.2f} MB")

Average size of parquet files: 24.40 MB


## Question 2: 

**Yellow November 2025** (1 point)

- 6MB
- **25MB**
- 75MB
- 100MB

In [17]:
from pyspark.sql import functions as F

df.withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15') \
    .count()

162604

## Question 3: 

**Count records** (1 point)

- 62,610
- 102,340
- **162,604**
- 225,768

In [20]:
# Answer to Question 4
from pyspark.sql import functions as F

# What: Calculating duration in hours
# Why: Difference in unix_timestamp (seconds) divided by 3600 (seconds in an hour)
df.withColumn('duration_hrs', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .select(F.max('duration_hrs')) \
    .show()

+-----------------+
|max(duration_hrs)|
+-----------------+
|90.64666666666666|
+-----------------+



#### Question 4: 

**Longest trip** (1 point)

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- **90.6**
- 134.5

In [21]:

spark.sparkContext.uiWebUrl

'http://0.0.0.0:4040'

## Question 5: User Interface (1 point)

- 80
- 443
- **4040**
- 8080

In [23]:
# Loading the taxi zone lookup table
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")

In [24]:
# Joining the Yellow Taxi data with the Zone lookup
df_result = df.join(df_zones, df.PULocationID == df_zones.LocationID)

In [28]:
# Grouping by Zone and counting the number of records
df_result.groupBy('Zone') \
    .count() \
    .orderBy('count', ascending=True) \
    .show(1) # Showing the single lowest

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+
only showing top 1 row



## Question 6: Least frequent pickup location zone (1 point)

- **Governor's Island/Ellis Island/Liberty Island**
- Arden Heights
- Rikers Island
- Jamaica Bay